# Mini End-to-End Data Science Project: Employee Attrition Prediction
### Internship Track: Week 4 – Task 3
**Author:** Sayantani Adhikary  
**Workflow:** Data Collection → Cleaning → EDA → Modeling → Evaluation → Insights


## 1. Data Collection
Load the raw HR dataset from `data/raw_dataset.csv`.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

sns.set_theme(style="whitegrid")
df_raw = pd.read_csv('../data/raw_dataset.csv')
print(f"Raw Shape: {df_raw.shape}")
df_raw.head()


## 2. Data Cleaning
1. Remove duplicate `EmployeeID` rows.
2. Standardize `Department` casing (`sales` -> `Sales`, `SUPPORT` -> `Support`).
3. Impute missing `MonthlyIncome` with the median (`5400.0`).
4. Export the final cleaned dataset to `data/cleaned_dataset.csv`.


In [1]:
df_clean = df_raw.drop_duplicates(subset=['EmployeeID']).copy()
df_clean['Department'] = df_clean['Department'].str.strip().str.title().replace({'Hr': 'HR'})
df_clean['MonthlyIncome'] = df_clean['MonthlyIncome'].fillna(df_clean['MonthlyIncome'].median()).round(2)

df_clean.to_csv('../data/cleaned_dataset.csv', index=False)
print(f"Cleaned Shape: {df_clean.shape} | Nulls remaining: {df_clean.isnull().sum().sum()}")


## 3. Exploratory Data Analysis (EDA)
Visualize how `OverTime` and `MonthlyIncome` relate to `Attrition`.


In [1]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.countplot(x='OverTime', hue='Attrition', data=df_clean, palette='Set2', ax=axes[0])
axes[0].set_title('Attrition Count by OverTime Status', fontweight='bold')

sns.boxplot(x='Attrition', y='MonthlyIncome', data=df_clean, palette='Set1', ax=axes[1])
axes[1].set_title('Monthly Income Distribution by Attrition', fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Modeling & Evaluation
Encode categorical columns, scale numerical features, and compare **Logistic Regression** vs. **Random Forest**.


In [1]:
X = df_clean.drop(columns=['EmployeeID', 'Attrition'])
y = df_clean['Attrition']

X['OverTime'] = X['OverTime'].map({'Yes': 1, 'No': 0})
X = pd.get_dummies(X, columns=['Department'], drop_first=True, dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
}

results = []
for name, m in models.items():
    m.fit(X_train_s, y_train)
    preds = m.predict(X_test_s)
    results.append({
        'Model': name,
        'Accuracy (%)': round(accuracy_score(y_test, preds) * 100, 2),
        'Precision (%)': round(precision_score(y_test, preds, zero_division=0) * 100, 2),
        'Recall (%)': round(recall_score(y_test, preds, zero_division=0) * 100, 2),
        'F1-Score': round(f1_score(y_test, preds, zero_division=0), 4)
    })

print(pd.DataFrame(results).to_string(index=False))


## 5. Key Insights
1. **Overtime Burnout**: Employees working overtime exhibit sharply higher attrition rates.
2. **Compensation & Early Tenure**: Lower-income staff in their first 1–2 years account for nearly all departures.
